1.Data Ingestion

In [5]:
import os
import zipfile
import gdown

In [6]:
SOURCE_URL = "https://drive.google.com/file/d/1z0mreUtRmR-P-magILsDR3T7M6IkGXtY/view?usp=sharing"
LOCAL_DATA_FILE = "artifacts/data_ingestion/data.zip"
UNZIP_DIR = "artifacts/data_ingestion/"

In [7]:
os.chdir("../")
os.makedirs("artifacts/data_ingestion", exist_ok=True)
print(f" Directory created: artifacts/data_ingestion")

 Directory created: artifacts/data_ingestion


In [8]:
file_id = SOURCE_URL.split("/d/")[1].split("/")[0]
download_url = f"https://drive.google.com/uc?id={file_id}"

In [9]:
gdown.download(download_url, LOCAL_DATA_FILE, quiet=False)

print(f"Downloaded to: {LOCAL_DATA_FILE}")

Downloading...
From (original): https://drive.google.com/uc?id=1z0mreUtRmR-P-magILsDR3T7M6IkGXtY
From (redirected): https://drive.google.com/uc?id=1z0mreUtRmR-P-magILsDR3T7M6IkGXtY&confirm=t&uuid=78e4bbe0-3655-433f-abc0-5cfae6facc8a
To: c:\Users\mazen\OneDrive\Desktop\artifacts\data_ingestion\data.zip
100%|██████████| 49.0M/49.0M [00:21<00:00, 2.29MB/s]

Downloaded to: artifacts/data_ingestion/data.zip


In [10]:
print(f" Extracting {LOCAL_DATA_FILE} ...")
with zipfile.ZipFile(LOCAL_DATA_FILE, "r") as zip_ref:
    zip_ref.extractall(UNZIP_DIR)
print(f" Extracted to: {UNZIP_DIR}")


 Extracting artifacts/data_ingestion/data.zip ...
 Extracted to: artifacts/data_ingestion/


In [11]:
import os

print("CURRENT DIR:", os.getcwd())
print("FILES HERE:", os.listdir("artifacts/data_ingestion"))

CURRENT DIR: c:\Users\mazen\OneDrive\Desktop
FILES HERE: ['Chest-CT-Scan-data', 'data.zip']


In [12]:
%pwd

'c:\\Users\\mazen\\OneDrive\\Desktop'

02_prepare_base_model

In [13]:
import tensorflow as tf


c:\Users\mazen\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\mazen\AppData\Local\Programs\Python\Python3

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\mazen\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\mazen\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\mazen\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\mazen\AppData\Local\Programs\Python\Python310\lib\site-packages\trait

AttributeError: _ARRAY_API not found

In [ ]:
ROOT_DIR         = "artifacts/prepare_base_model"
BASE_MODEL_PATH  = "artifacts/prepare_base_model/base_model.h5"
UPDATED_MODEL_PATH = "artifacts/prepare_base_model/base_model_updated.h5"

In [15]:

IMAGE_SIZE    = [224, 224, 3]
LEARNING_RATE = 0.01
# Remove the last classification part of the model.
INCLUDE_TOP   = False
# Without ImageNet:
# Your CNN starts with random weights → learns slowly
WEIGHTS       = "imagenet"
CLASSES       = 2

In [16]:
os.makedirs(ROOT_DIR, exist_ok=True)
print(f"✅ Directory created: {ROOT_DIR}")


✅ Directory created: artifacts/prepare_base_model


In [17]:
model = tf.keras.applications.VGG16(
    input_shape=IMAGE_SIZE,
    weights=WEIGHTS,
    include_top=INCLUDE_TOP
)

In [18]:
import os
%cd ..
base = "artifacts/data_ingestion/Chest-CT-Scan-data"

folders = [
    os.path.join(base, "adenocarcinoma"),
    os.path.join(base, "normal")
]

images = []

for folder in folders:
    images += [
        f for f in os.listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))
    ]

print("Total images:", len(images))

c:\Users\mazen\OneDrive
Total images: 343


c:\Users\mazen\AppData\Local\Programs\Python\Python310\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [19]:
model.save(BASE_MODEL_PATH)
print(f"✅ Base model saved to: {BASE_MODEL_PATH}")


✅ Base model saved to: artifacts/prepare_base_model/base_model.h5


In [20]:
for layer in model.layers:
    layer.trainable = False

In [21]:
flatten = tf.keras.layers.Flatten()(model.output)
output  = tf.keras.layers.Dense(units=CLASSES, activation="softmax")(flatten)

In [22]:
full_model = tf.keras.models.Model(inputs=model.input, outputs=output)


In [23]:
full_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"]
)

In [24]:
full_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │        50,178 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,764,866 (56.32 MB)

 Trainable params: 50,178 (196.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [27]:
full_model.save(UPDATED_MODEL_PATH)
print(f"✅ Updated model saved to: {UPDATED_MODEL_PATH}")

✅ Updated model saved to: artifacts/prepare_base_model/base_model_updated.h5


In [30]:
print(os.listdir("artifacts/prepare_base_model"))

['base_model.h5', 'base_model_updated.h5']
